# Experiment 3 — Product in Action

**Claim:** AnyUp3D enables practical video object segmentation on real videos with temporally stable masks.

**Pipeline:** Video Swin-T → AnyUp3D (or bilinear) → binary linear head → per-frame mask

**Metric:** J&F score (DAVIS 2017 val split)

**Head training:** DAVIS 2017 train split, binary foreground/background

**Val features:** reused from Exp 2 Swin cache

**Seeds:** 3 (mean ± std reported, paired t-test)

In [ ]:
# ── Cell 1 · Installs & repo setup ─────────────────────────────────────────
import subprocess, sys

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

# Clone AnyUp repo (skip if already present)
subprocess.run(
    ["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
    check=True
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# DAVIS evaluation package
subprocess.run(
    ["git", "clone", "https://github.com/davisvideochallenge/davis2017-evaluation.git",
     "/content/davis2017-evaluation"],
    check=True
)
subprocess.run(
    [sys.executable, "setup.py", "install"],
    cwd="/content/davis2017-evaluation",
    check=True
)

# natten — required by AnyUp cross-attention
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "natten",
     "--find-links", "https://shi-labs.com/natten/wheels/cu118/torch2.0.0/index.html"],
    check=False   # fall through; natten may already be cached
)

print("Cell 1 done.")

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
import os

# ── Paths ───────────────────────────────────────────────────────────────────
CKPT_ANYUP3D = "/content/drive/MyDrive/anyup3d/checkpoints/run_01/anyup3d_step10000.pth"
DAVIS_ZIP    = "/content/drive/MyDrive/anyup3d/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT   = "/content/DAVIS"
OUTPUT_DIR   = "/content/drive/MyDrive/anyup3d/results/exp3"

# Reuse Exp 2 val cache — each file: {clip}.pt → {feats:(T,768,7,7), guides:(T,3,224,224)}
VAL_CACHE_DIR   = "/content/drive/MyDrive/anyup3d/results/exp2/swin_cache"
TRAIN_CACHE_DIR = os.path.join(OUTPUT_DIR, "train_swin_cache")
MASK_DIR        = os.path.join(OUTPUT_DIR, "masks")      # predicted mask PNGs
HEAD_CKPT_DIR   = os.path.join(OUTPUT_DIR, "head_ckpts") # one per seed
STILLS_DIR      = os.path.join(OUTPUT_DIR, "stills")
VIDEO_DIR       = os.path.join(OUTPUT_DIR, "videos")

for d in [OUTPUT_DIR, TRAIN_CACHE_DIR, MASK_DIR, HEAD_CKPT_DIR, STILLS_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Model ───────────────────────────────────────────────────────────────────
IMG_SIZE  = 224   # ← guide and output resolution; FEAT_SIZE depends on this
FEAT_SIZE = 7     # ← Swin-T norm output spatial size at IMG_SIZE=224
FEAT_DIM  = 768   # ← Swin-T channel dim; also head input channels

# ── Swin extraction ─────────────────────────────────────────────────────────
FRAME_STEP = 12   # ← sample every 12th frame (24fps→2fps); more frames = more GPU mem

# ── Head training ───────────────────────────────────────────────────────────
HEAD_LR     = 1e-3  # ← linear head learning rate
HEAD_EPOCHS = 20    # ← epochs over DAVIS train split; reduce to 10 if slow
SEEDS       = [0, 1, 2]  # ← three seeds for mean±std

# ── J&F clips ───────────────────────────────────────────────────────────────
N_VISUAL_CLIPS = 4  # ← clips to include in side-by-side stills figure

print("Config loaded.")
print(f"  IMG_SIZE={IMG_SIZE}, FEAT_SIZE={FEAT_SIZE}, FEAT_DIM={FEAT_DIM}")
print(f"  FRAME_STEP={FRAME_STEP}, HEAD_EPOCHS={HEAD_EPOCHS}, SEEDS={SEEDS}")

In [ ]:
# ── Cell 3 · Imports ────────────────────────────────────────────────────────
import sys, os, random
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 4 · Model loading ──────────────────────────────────────────────────
#
# TStage must be defined before torch.load — checkpoint pickle needs it.
class TStage:
    def __init__(self, t=None, start_step=None, batch_size=None):
        self.t          = t
        self.start_step = start_step
        self.batch_size = batch_size

# ── AnyUp3D ─────────────────────────────────────────────────────────────────
from anyup.model import AnyUp

model_3d = AnyUp(
    input_dim       = 3,
    qk_dim          = 128,
    kernel_size     = 1,
    kernel_size_lfu = 5,   # ← omitting causes 73 missing keys
    window_ratio    = 0.1,
    num_heads       = 4,
    t_k             = 1,
).to(device).eval()

ckpt = torch.load(CKPT_ANYUP3D, map_location="cpu", weights_only=False)  # weights_only=False required for TStage
sd   = ckpt["anyup"]   # ← key is "anyup", not "model" or "state_dict"
missing, unexpected = model_3d.load_state_dict(sd, strict=False)
print(f"AnyUp3D loaded — missing: {len(missing)}, unexpected: {len(unexpected)}")
if missing:
    print(f"  First 5 missing: {missing[:5]}")

# ── AnyUp2D ─────────────────────────────────────────────────────────────────
from anyup.model_2d import AnyUp2D   # direct import; do NOT use torch.hub.load

model_2d = AnyUp2D().to(device).eval()
sd_2d = torch.hub.load_state_dict_from_url(
    "https://github.com/wimmerth/anyup/releases/download/checkpoint/anyup_paper.pth",
    map_location=device
)
model_2d.load_state_dict(sd_2d)
print("AnyUp2D loaded.")

# ── Video Swin-T ─────────────────────────────────────────────────────────────
_feat = {}

def _hook_fn(module, inp, out):
    # Swin norm outputs (B, T, H, W, C) — permute to (B, C, T, H, W)
    _feat["norm"] = out.detach().permute(0, 4, 1, 2, 3).contiguous()

backbone = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1).to(device).eval()
backbone.norm.register_forward_hook(_hook_fn)
for p in backbone.parameters():
    p.requires_grad_(False)
print("Swin-T loaded.")

# ── Preprocessing ────────────────────────────────────────────────────────────
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
frame_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
guide_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
# ── Cell 5 · DAVIS setup ────────────────────────────────────────────────────
import subprocess

# Extract zip if DAVIS not yet unpacked
if not os.path.isdir(DAVIS_ROOT):
    print("Extracting DAVIS zip...")
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)
    print("Extracted.")
else:
    print("DAVIS already extracted.")

IMG_DIR  = os.path.join(DAVIS_ROOT, "JPEGImages",  "480p")
ANN_DIR  = os.path.join(DAVIS_ROOT, "Annotations", "480p")
SET_DIR  = os.path.join(DAVIS_ROOT, "ImageSets",   "2017")

def load_clip_list(split):   # split: "train" or "val"
    path = os.path.join(SET_DIR, f"{split}.txt")
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_clips = load_clip_list("train")
val_clips   = load_clip_list("val")
print(f"Train clips: {len(train_clips)}, Val clips: {len(val_clips)}")

def get_frame_paths(clip_name):
    """Sorted list of frame paths for a clip."""
    return sorted(Path(IMG_DIR, clip_name).glob("*.jpg"))

def get_ann_paths(clip_name):
    """Sorted list of annotation paths (palette PNGs) for a clip."""
    return sorted(Path(ANN_DIR, clip_name).glob("*.png"))

def sample_frame_indices(total_frames):
    """Sample every FRAME_STEP-th frame; skip clip if fewer than 2 result."""
    idxs = list(range(0, total_frames, FRAME_STEP))  # ← FRAME_STEP controls T and memory
    return idxs if len(idxs) >= 2 else []

def load_binary_mask(ann_path):
    """Load annotation PNG → float32 binary mask (foreground=1, background=0)."""
    ann = np.array(Image.open(ann_path))  # values 0..N_objects
    return (ann > 0).astype(np.float32)   # any object ID > 0 → foreground

# Quick sanity check
sample_frames = get_frame_paths(val_clips[0])
sample_idxs   = sample_frame_indices(len(sample_frames))
print(f"Sample clip '{val_clips[0]}': {len(sample_frames)} frames, "
      f"{len(sample_idxs)} sampled at step={FRAME_STEP}")

In [ ]:
# ── Cell 6 · Swin feature extraction — DAVIS train split ───────────────────
#
# Val features are already cached in VAL_CACHE_DIR from Exp 2.
# This cell builds the equivalent cache for the train split.
# Format: {clip}.pt → {"feats": (T, 768, 7, 7), "guides": (T, 3, 224, 224)}

@torch.no_grad()
def extract_swin_features(clip_name, frame_paths, idxs):
    """
    Returns feats (T, 768, 7, 7) and guides (T, 3, 224, 224).
    Swin requires T≥2; single-frame input is duplicated.
    """
    frames_norm  = []  # normalized for Swin
    frames_guide = []  # unnormalized RGB for AnyUp guide

    for i in idxs:
        img = Image.open(frame_paths[i]).convert("RGB")
        frames_norm.append(frame_transform(img))
        frames_guide.append(guide_transform(img))

    T = len(idxs)  # ← number of sampled frames; controls VRAM during Swin forward
    # (B=1, C=3, T, H, W) for Swin
    vol = torch.stack(frames_norm, dim=1).unsqueeze(0).to(device)  # (1, 3, T, IMG_SIZE, IMG_SIZE)

    if T < 2:  # Swin requires T≥2
        vol = vol.expand(-1, -1, 2, -1, -1)

    backbone(vol)                                   # triggers hook → _feat["norm"]
    feats = _feat["norm"].squeeze(0).cpu()          # (C, T_out, H_out, W_out)

    # Swin may pool T → 1 for short clips; expand back to match frame count
    T_out = feats.shape[1]
    if T_out == 1 and T > 1:
        feats = feats.expand(-1, T, -1, -1)         # broadcast across frames
    feats = feats.permute(1, 0, 2, 3).float()       # (T, C, H, W) — float32 for training

    guides = torch.stack(frames_guide)               # (T, 3, IMG_SIZE, IMG_SIZE)
    return feats, guides


print("Extracting Swin features for DAVIS train split...")
skip_count = 0
for clip in tqdm(train_clips):
    cache_path = os.path.join(TRAIN_CACHE_DIR, f"{clip}.pt")
    if os.path.exists(cache_path):   # safe to re-run after disconnection
        continue

    frame_paths = get_frame_paths(clip)
    idxs = sample_frame_indices(len(frame_paths))
    if not idxs:
        skip_count += 1
        continue

    feats, guides = extract_swin_features(clip, frame_paths, idxs)
    torch.save({"feats": feats, "guides": guides}, cache_path)

    # Memory: each (T, 768, 7, 7) is small but clear anyway
    del feats, guides
    torch.cuda.empty_cache()

print(f"Train cache done. Skipped {skip_count} clips with <2 sampled frames.")

In [ ]:
# ── Cell 7 · Binary segmentation head — training ───────────────────────────
#
# Head: Conv2d(FEAT_DIM, 1, 1) — one binary output per spatial location.
# Input: upsampled features (FEAT_DIM, IMG_SIZE, IMG_SIZE) per frame.
# Label: binary mask (IMG_SIZE, IMG_SIZE) — foreground=1, background=0.
# Loss: BCEWithLogitsLoss.
#
# The head is trained once per seed on the TRAIN split using AnyUp3D features.
# The SAME trained head is then evaluated on the VAL split for both conditions
# (AnyUp3D and bilinear), making the comparison fair.

def make_head():
    return nn.Conv2d(FEAT_DIM, 1, kernel_size=1).to(device)  # ← FEAT_DIM: change if backbone changes


@torch.no_grad()
def upsample_3d(feats_TCHW, guides_T3HW):
    """
    Run AnyUp3D on one clip's features.
    feats_TCHW:  (T, 768, FEAT_SIZE, FEAT_SIZE)
    guides_T3HW: (T, 3,   IMG_SIZE,  IMG_SIZE)
    Returns: (T, 768, IMG_SIZE, IMG_SIZE)
    """
    T = feats_TCHW.shape[0]   # ← T controls VRAM; large T may OOM on T4
    feat_vol  = feats_TCHW.unsqueeze(0).to(device)   # (1, 768, T, 7,   7)
    guide_vol = guides_T3HW.unsqueeze(0).to(device)  # (1, 3,  T, 224, 224)
    out = model_3d(guide_vol, feat_vol)               # guide first — NOT (feat, guide)
    return out.squeeze(0).permute(1, 0, 2, 3)         # (T, 768, 224, 224)


@torch.no_grad()
def upsample_bilinear(feats_TCHW):
    """
    Bilinear upsample each frame independently.
    Returns: (T, 768, IMG_SIZE, IMG_SIZE)
    """
    return F.interpolate(
        feats_TCHW.to(device),                   # (T, 768, 7, 7)
        size=(IMG_SIZE, IMG_SIZE),               # ← IMG_SIZE: change if resolution changes
        mode="bilinear",
        align_corners=False
    )


def get_train_annotations(clip):
    """
    Load sampled binary masks for a train clip, resized to IMG_SIZE.
    Returns: (T, IMG_SIZE, IMG_SIZE) float32 tensor.
    """
    frame_paths = get_frame_paths(clip)
    ann_paths   = get_ann_paths(clip)
    idxs = sample_frame_indices(len(frame_paths))
    masks = []
    for i in idxs:
        if i >= len(ann_paths):
            continue
        m = load_binary_mask(ann_paths[i])           # (H_orig, W_orig)
        m = cv2.resize(m, (IMG_SIZE, IMG_SIZE),      # ← IMG_SIZE: change with resolution
                       interpolation=cv2.INTER_NEAREST)
        masks.append(torch.from_numpy(m))
    if not masks:
        return None
    return torch.stack(masks)  # (T, IMG_SIZE, IMG_SIZE)


# ── Training loop ────────────────────────────────────────────────────────────
head_checkpoints = {}  # seed → state_dict

for seed in SEEDS:
    print(f"\n── Seed {seed} ──────────────────────────────────")
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    head = make_head()
    opt  = torch.optim.Adam(head.parameters(), lr=HEAD_LR)  # ← HEAD_LR: tune if diverges
    loss_fn = nn.BCEWithLogitsLoss()

    # Collect valid train clips (those with cached features and annotations)
    valid_train = [
        c for c in train_clips
        if os.path.exists(os.path.join(TRAIN_CACHE_DIR, f"{c}.pt"))
    ]
    print(f"  Train clips with cache: {len(valid_train)}")

    for epoch in range(HEAD_EPOCHS):  # ← HEAD_EPOCHS: reduce to 10 if slow
        head.train()
        random.shuffle(valid_train)
        epoch_loss = 0.0
        n_batches  = 0

        for clip in valid_train:
            cache = torch.load(
                os.path.join(TRAIN_CACHE_DIR, f"{clip}.pt"),
                map_location="cpu", weights_only=False
            )
            feats_TCHW  = cache["feats"]   # (T, 768, 7, 7)
            guides_T3HW = cache["guides"]  # (T, 3, 224, 224)

            masks = get_train_annotations(clip)   # (T, 224, 224) or None
            if masks is None:
                continue

            T = min(feats_TCHW.shape[0], masks.shape[0])  # align T if mismatch
            if T == 0:
                continue
            feats_TCHW  = feats_TCHW[:T]
            guides_T3HW = guides_T3HW[:T]
            masks       = masks[:T].to(device)  # (T, 224, 224)

            # Upsample with AnyUp3D (no grad needed for frozen model)
            with torch.no_grad():
                upsampled = upsample_3d(feats_TCHW, guides_T3HW)  # (T, 768, 224, 224)

            # Head forward: process all T frames as a batch
            logits = head(upsampled).squeeze(1)  # (T, 224, 224)
            loss   = loss_fn(logits, masks)

            opt.zero_grad()
            loss.backward()
            opt.step()

            epoch_loss += loss.item()
            n_batches  += 1

            # Free GPU memory after each clip — critical to avoid OOM on T4
            del upsampled, logits, loss
            torch.cuda.empty_cache()

        avg = epoch_loss / max(n_batches, 1)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:02d}/{HEAD_EPOCHS} — loss: {avg:.4f}")

    # Save head checkpoint for this seed
    ckpt_path = os.path.join(HEAD_CKPT_DIR, f"head_seed{seed}.pth")
    torch.save(head.state_dict(), ckpt_path)
    head_checkpoints[seed] = ckpt_path
    print(f"  Head saved → {ckpt_path}")

print("\nHead training complete.")

In [ ]:
# ── Cell 8 · Inference on val split — save mask PNGs ───────────────────────
#
# For each (seed × condition × clip × frame), save a binary mask PNG.
# Directory structure expected by davis2017-evaluation:
#   MASK_DIR/{method}_seed{s}/{clip}/{frame:05d}.png
# where method ∈ {"anyup3d", "bilinear"}.
#
# Mask PNG format: uint8, single channel (or palette), values 0 or 1.
# The davis eval package converts to uint8 internally.

def save_mask_png(logit_HW, out_path):
    """Threshold sigmoid output → binary PNG. logit_HW: (H, W) on any device."""
    prob = torch.sigmoid(logit_HW.cpu()).numpy()
    mask = (prob > 0.5).astype(np.uint8) * 255   # 0 or 255 for DAVIS eval
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    Image.fromarray(mask, mode="L").save(out_path)


@torch.no_grad()
def run_inference_clip(clip, cache_dir, head, condition):
    """
    condition: "anyup3d" or "bilinear"
    Returns list of frame stem names (for filename alignment).
    """
    cache_path = os.path.join(cache_dir, f"{clip}.pt")
    if not os.path.exists(cache_path):
        return []

    cache = torch.load(cache_path, map_location="cpu", weights_only=False)
    feats_TCHW  = cache["feats"]   # (T, 768, 7, 7)
    guides_T3HW = cache["guides"]  # (T, 3, 224, 224)
    T = feats_TCHW.shape[0]        # ← T varies per clip

    # Upsample features
    if condition == "anyup3d":
        upsampled = upsample_3d(feats_TCHW, guides_T3HW)   # (T, 768, 224, 224)
    else:  # bilinear
        upsampled = upsample_bilinear(feats_TCHW)           # (T, 768, 224, 224)

    head.eval()
    logits = head(upsampled).squeeze(1)   # (T, 224, 224)

    # Retrieve frame names to align mask filenames with DAVIS ground truth
    frame_paths = get_frame_paths(clip)
    idxs = sample_frame_indices(len(frame_paths))
    frame_stems = [Path(frame_paths[i]).stem for i in idxs[:T]]

    del upsampled
    torch.cuda.empty_cache()
    return logits, frame_stems


# Check which val clips have Exp 2 cache available
cached_val_clips = [
    c for c in val_clips
    if os.path.exists(os.path.join(VAL_CACHE_DIR, f"{c}.pt"))
]
print(f"Val clips with cache: {len(cached_val_clips)} / {len(val_clips)}")

for seed in SEEDS:
    print(f"\n── Seed {seed} inference ────────────────────────")
    head = make_head()
    head.load_state_dict(torch.load(head_checkpoints[seed], map_location=device))
    head.eval()

    for condition in ["anyup3d", "bilinear"]:
        method_dir = os.path.join(MASK_DIR, f"{condition}_seed{seed}")

        for clip in tqdm(cached_val_clips, desc=f"{condition}"):
            result = run_inference_clip(clip, VAL_CACHE_DIR, head, condition)
            if not result:
                continue
            logits, frame_stems = result

            clip_mask_dir = os.path.join(method_dir, clip)
            for t, stem in enumerate(frame_stems):
                save_mask_png(logits[t], os.path.join(clip_mask_dir, f"{stem}.png"))

            del logits
            torch.cuda.empty_cache()

print("\nInference complete. Mask PNGs saved.")

In [ ]:
# ── Cell 9 · J&F evaluation ─────────────────────────────────────────────────
#
# Uses the official davis2017-evaluation package.
# We call the Python API directly (avoids subprocess argument quoting issues).

from davis2017.evaluation import DAVISEvaluation

results_jf = {}   # (condition, seed) → {"J": float, "F": float, "JF": float}

for seed in SEEDS:
    for condition in ["anyup3d", "bilinear"]:
        method_dir = os.path.join(MASK_DIR, f"{condition}_seed{seed}")

        # davis2017-evaluation expects results_path to contain one folder per clip
        evaluator = DAVISEvaluation(
            davis_root   = DAVIS_ROOT,
            task         = "semi-supervised",
            gt_set       = "val",
            sequences    = "all",
        )
        metrics_per_sequence, metrics_summary = evaluator.evaluate(method_dir)

        J_mean = metrics_summary["J"]["M"]
        F_mean = metrics_summary["F"]["M"]
        JF_mean = (J_mean + F_mean) / 2.0
        results_jf[(condition, seed)] = {"J": J_mean, "F": F_mean, "JF": JF_mean}
        print(f"  {condition} seed={seed} → J={J_mean:.3f}, F={F_mean:.3f}, J&F={JF_mean:.3f}")

print("\nRaw J&F results:", results_jf)

In [ ]:
# ── Cell 10 · Statistics ────────────────────────────────────────────────────
from scipy import stats

def summarize(condition):
    jf_vals = [results_jf[(condition, s)]["JF"] for s in SEEDS]
    return np.mean(jf_vals), np.std(jf_vals), jf_vals

mean_3d,  std_3d,  jf_3d  = summarize("anyup3d")
mean_bil, std_bil, jf_bil = summarize("bilinear")

# Paired t-test across 3 seeds (J&F)
t_stat, p_val = stats.ttest_rel(jf_3d, jf_bil)

print("\n── Table: J&F Score (DAVIS 2017 val) ──────────────────")
print(f"{'Method':<20} {'J&F (mean±std)':>20}")
print("-" * 42)
print(f"{'Bilinear':<20} {mean_bil:.3f} ± {std_bil:.3f}")
print(f"{'AnyUp3D':<20} {mean_3d:.3f}  ± {std_3d:.3f}")
print("-" * 42)
print(f"Paired t-test p-value: {p_val:.4f}  (t={t_stat:.3f})")
print(f"Result: {'significant' if p_val < 0.05 else 'not significant'} at α=0.05")

# Per-seed breakdown
print("\nPer-seed J&F:")
for s in SEEDS:
    j3, j2 = results_jf[("anyup3d",s)]["JF"], results_jf[("bilinear",s)]["JF"]
    delta = j3 - j2
    print(f"  Seed {s}: AnyUp3D={j3:.3f}, Bilinear={j2:.3f}, Δ={delta:+.3f}")

# Save summary CSV
import csv
csv_path = os.path.join(OUTPUT_DIR, "jf_results.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["method", "seed", "J", "F", "JF"])
    for (cond, seed), vals in results_jf.items():
        w.writerow([cond, seed, f"{vals['J']:.4f}", f"{vals['F']:.4f}", f"{vals['JF']:.4f}"])
    w.writerow(["anyup3d", "MEAN", "", "", f"{mean_3d:.4f}"])
    w.writerow(["anyup3d", "STD",  "", "", f"{std_3d:.4f}"])
    w.writerow(["bilinear","MEAN", "", "", f"{mean_bil:.4f}"])
    w.writerow(["bilinear","STD",  "", "", f"{std_bil:.4f}"])
    w.writerow(["p-value", "", "", "", f"{p_val:.4f}"])
print(f"Saved → {csv_path}")

In [ ]:
# ── Cell 11 · Visual outputs ────────────────────────────────────────────────
#
# Produces:
#   (a) Side-by-side stills figure: N_VISUAL_CLIPS clips × 3 cols
#       [original frame | bilinear mask overlay | AnyUp3D mask overlay]
#   (b) One full comparison video (mp4) for the best clip
#
# Uses seed=0 masks for visuals.

VISUAL_SEED = 0   # ← which seed's masks to use for figures

def load_png_mask(mask_png_path):
    """Load saved mask PNG → float32 numpy (H, W) in [0, 1]."""
    m = np.array(Image.open(mask_png_path).convert("L"), dtype=np.float32)
    return m / 255.0

def overlay_mask(frame_np_HWC, mask_HW, color=(0, 1, 0), alpha=0.45):
    """
    Blend binary mask onto RGB frame.
    frame_np_HWC: float32 (H, W, 3) in [0,1]
    mask_HW:      float32 (H, W) in [0,1]
    """
    overlay = frame_np_HWC.copy()
    fg = mask_HW > 0.5
    for c, v in enumerate(color):
        overlay[fg, c] = overlay[fg, c] * (1 - alpha) + v * alpha
    return overlay

def load_frame_np(frame_path):
    img = Image.open(frame_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.array(img, dtype=np.float32) / 255.0


# Pick representative clips (first N_VISUAL_CLIPS with cache)
visual_clips = cached_val_clips[:N_VISUAL_CLIPS]  # ← N_VISUAL_CLIPS controls figure rows

# ── (a) Side-by-side stills ──────────────────────────────────────────────────
fig, axes = plt.subplots(
    N_VISUAL_CLIPS, 3,
    figsize=(12, 3.5 * N_VISUAL_CLIPS)  # ← N_VISUAL_CLIPS also controls figure height
)
if N_VISUAL_CLIPS == 1:
    axes = [axes]

col_titles = ["Original frame", "Bilinear mask", "AnyUp3D mask"]

for row, clip in enumerate(visual_clips):
    frame_paths = get_frame_paths(clip)
    idxs = sample_frame_indices(len(frame_paths))
    if not idxs:
        continue

    # Pick middle frame for the still
    mid_i  = idxs[len(idxs) // 2]
    stem   = Path(frame_paths[mid_i]).stem
    frame  = load_frame_np(frame_paths[mid_i])

    mask_bil_path = os.path.join(MASK_DIR, f"bilinear_seed{VISUAL_SEED}", clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, f"anyup3d_seed{VISUAL_SEED}",  clip, f"{stem}.png")

    axes[row][0].imshow(frame)
    axes[row][0].set_ylabel(clip, fontsize=9)

    if os.path.exists(mask_bil_path):
        m_bil = load_png_mask(mask_bil_path)
        m_bil = cv2.resize(m_bil, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        axes[row][1].imshow(overlay_mask(frame, m_bil, color=(1, 0.4, 0)))

    if os.path.exists(mask_3d_path):
        m_3d = load_png_mask(mask_3d_path)
        m_3d = cv2.resize(m_3d, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        axes[row][2].imshow(overlay_mask(frame, m_3d, color=(0, 0.8, 0)))

    for ax in axes[row]:
        ax.axis("off")

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight="bold")

plt.tight_layout()
stills_path = os.path.join(STILLS_DIR, "side_by_side.png")
plt.savefig(stills_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Stills saved → {stills_path}")


# ── (b) Comparison video for best clip ──────────────────────────────────────
# Pick the clip where AnyUp3D vs bilinear delta J&F is largest (from per-clip
# metrics if available; otherwise just use first visual clip).
best_clip = visual_clips[0]  # fallback — swap for highest-delta clip if known

frame_paths = get_frame_paths(best_clip)
idxs = sample_frame_indices(len(frame_paths))

video_path = os.path.join(VIDEO_DIR, f"{best_clip}_comparison.mp4")
fps        = 4   # ← output video FPS; low because we sampled at 2fps from 24fps source

# Each output frame is 3 panels wide: IMG_SIZE × (3 × IMG_SIZE)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(video_path, fourcc, fps,
                         (IMG_SIZE * 3, IMG_SIZE))  # ← width=3×IMG_SIZE; height=IMG_SIZE

for i, src_i in enumerate(idxs):
    stem  = Path(frame_paths[src_i]).stem
    frame = load_frame_np(frame_paths[src_i])   # (H, W, 3) float [0,1]

    mask_bil_path = os.path.join(MASK_DIR, f"bilinear_seed{VISUAL_SEED}", best_clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, f"anyup3d_seed{VISUAL_SEED}",  best_clip, f"{stem}.png")

    panel_orig = (frame * 255).astype(np.uint8)

    if os.path.exists(mask_bil_path):
        m = cv2.resize(load_png_mask(mask_bil_path), (IMG_SIZE, IMG_SIZE),
                       interpolation=cv2.INTER_NEAREST)
        panel_bil = (overlay_mask(frame, m, color=(1, 0.4, 0)) * 255).astype(np.uint8)
    else:
        panel_bil = panel_orig.copy()

    if os.path.exists(mask_3d_path):
        m = cv2.resize(load_png_mask(mask_3d_path), (IMG_SIZE, IMG_SIZE),
                       interpolation=cv2.INTER_NEAREST)
        panel_3d = (overlay_mask(frame, m, color=(0, 0.8, 0)) * 255).astype(np.uint8)
    else:
        panel_3d = panel_orig.copy()

    row_frame = np.concatenate([panel_orig, panel_bil, panel_3d], axis=1)  # (H, 3W, 3)
    writer.write(cv2.cvtColor(row_frame, cv2.COLOR_RGB2BGR))

writer.release()
print(f"Video saved → {video_path}")
print("\n── Experiment 3 complete. ──")